In [ ]:
# Libraries
import hashlib, json, re, unicodedata
from pathlib import Path
from collections import defaultdict
from difflib import SequenceMatcher
import pandas as pd


In [ ]:
# Text, numeric, ID and CSV helpers
def txt(x):
    return '' if x is None else str(x).strip()

def norm(x):
    s=unicodedata.normalize('NFKD',txt(x)).casefold()
    return ' '.join(re.sub(r'[^\w]+',' ',''.join(c for c in s if not unicodedata.combining(c))).split())

def terms(x):
    return list(dict.fromkeys(norm(t).replace(' ','_') for t in txt(x).split(',') if norm(t) not in {'','unknown','nan','none','null','n a'}))

def num(x):
    try:
        v=float(txt(x).replace(',','.'))
        return v if pd.notna(v) and abs(v)!=float('inf') else None
    except ValueError:
        return None

def fid(url):
    m=re.search(r'-(\d+)\.html(?:[?#].*)?$',txt(url),re.I)
    return 'fr:'+m.group(1) if m else ''

def key(name,brand):
    n,b=norm(name),norm(brand)
    return n[len(b)+1:] if b and n.startswith(b+' ') else n

def csv(df,path):
    df.to_csv(path,index=False,encoding='utf-8-sig',lineterminator='\n')


In [ ]:
# Preprocessing pipeline
def run(fdir,apath,out):
    out.mkdir(parents=True,exist_ok=True)
    audit=out/'audit'; audit.mkdir(exist_ok=True)
    cpath,rpath=fdir/'fra_cleaned.csv',fdir/'fra_perfumes.csv'
    sources=[dict(filename=p.name,size_bytes=p.stat().st_size,sha256=hashlib.sha256(p.read_bytes()).hexdigest()) for p in [cpath,rpath,apath]]
    try:
        clean=pd.read_csv(cpath,sep=';',encoding='utf-8-sig',dtype=str,keep_default_na=False); encoding='utf-8-sig'
    except UnicodeDecodeError:
        clean=pd.read_csv(cpath,sep=';',encoding='cp1252',dtype=str,keep_default_na=False); encoding='cp1252'
    raw=pd.read_csv(rpath,dtype=str,keep_default_na=False,encoding='utf-8-sig')
    records=json.loads(apath.read_text(encoding='utf-8-sig'))
    raw['source_row']=range(1,len(raw)+1); raw['perfume_id']=raw.url.map(fid)
    raw['rating_count_numeric']=raw['Rating Count'].map(num).fillna(-1)
    csv(raw[raw.duplicated('perfume_id',keep=False)].drop(columns='Description'),audit/'fragrantica_raw_duplicate_ids.csv')
    raw_unique=raw.sort_values(['rating_count_numeric','source_row'],ascending=[False,True]).drop_duplicates('perfume_id')
    lookup=raw_unique.set_index('perfume_id').to_dict('index')
    items=[]
    for i,r in enumerate(clean.to_dict('records'),1):
        pid=fid(r['url']); assert pid
        top,mid,base=[terms(r[c]) for c in ['Top','Middle','Base']]
        accords=list(dict.fromkeys(t for n in range(1,6) for t in terms(r.get('mainaccord'+str(n),''))))
        tokens=['note_'+t for t in dict.fromkeys(top+mid+base)]+['accord_'+t for t in accords]
        extra=lookup.get(pid,{})
        items.append(dict(perfume_id=pid,perfume_name=r['Perfume'],brand=r['Brand'],name_normalized=norm(r['Perfume']),brand_normalized=norm(r['Brand']),gender=r['Gender'],release_year=num(r['Year']),country=r['Country'],top_notes=json.dumps(top,ensure_ascii=False),middle_notes=json.dumps(mid,ensure_ascii=False),base_notes=json.dumps(base,ensure_ascii=False),main_accords=json.dumps(accords,ensure_ascii=False),content_text=' '.join(tokens),content_eligible=bool(tokens),avg_rating=num(r['Rating Value']),rating_count=num(r['Rating Count']),url=r['url'],raw_display_name=extra.get('Name',''),description=extra.get('Description',''),source_dataset=cpath.name,source_row=i,supplement_source_row=extra.get('source_row','')))
    content=pd.DataFrame(items); assert content.perfume_id.is_unique
    csv(content,out/'perfumes_content.csv')
    csv(raw_unique[~raw_unique.perfume_id.isin(content.perfume_id)][['perfume_id','Name','url','source_row']],audit/'fragrantica_outside_content_catalog.csv')
    reviews=[]
    for i,r in enumerate(records,1):
        parent=r.get('user_vote_on_perfume') or {}; v=parent.get('votes') or {}
        if not isinstance(v,dict): v={}
        p=r.get('production_specifications') or {}; s=r.get('scent_specifications') or {}; community=r.get('users_vote') or {}
        uid,pid=txt(r.get('user_id')),txt(r.get('perfume_id'))
        sr=txt(v.get('\u0631\u0627\u06cc\u062d\u0647')); rating=num(sr)
        if not uid or not pid: status='missing_id'
        elif not sr: status='missing_rating'
        elif rating is None: status='invalid_numeric_rating'
        elif rating==0: status='zero_unresolved'
        elif not 1<=rating<=10: status='out_of_range'
        else: status='observed_1_to_10'
        reviews.append(dict(user_id='atr:user:'+uid if uid else '',perfume_id='atr:'+pid if pid else '',perfume_name=txt(r.get('perfume_name')),brand=txt(r.get('perfume_brand')),release_year=num(p.get('release_year')),url=txt(r.get('perfume_url')),fragrance_group_raw=txt(s.get('fragrance_group')),nature_raw=txt(s.get('nature')),community_scent_rating=num(community.get('smell')),scent_rating_raw=sr,scent_rating=rating,longevity_rating_raw=txt(v.get('\u0645\u0627\u0646\u062f\u06af\u0627\u0631\u06cc')),sillage_rating_raw=txt(v.get('\u067e\u062e\u0634 \u0628\u0648')),bottle_rating_raw=txt(v.get('\u0637\u0631\u0627\u062d\u06cc \u0634\u06cc\u0634\u0647')),rating_status=status,comment_id=txt(parent.get('comment_id')),is_comment_answer=txt(parent.get('is_comment_answer')),review_text=txt(r.get('comment_text')),source_dataset=apath.name,source_row=i))
    review=pd.DataFrame(reviews); csv(review,out/'atrafshan_reviews_preserved.csv')
    eligible=review[review.rating_status.eq('observed_1_to_10')].copy()
    counts=eligible.groupby(['user_id','perfume_id']).scent_rating.nunique()
    conflict_pairs=set(counts[counts>1].index)
    conflict=eligible.apply(lambda r:(r.user_id,r.perfume_id) in conflict_pairs,axis=1)
    csv(eligible[conflict],audit/'conflicting_ratings.csv')
    ratings=eligible[~conflict].drop_duplicates(['user_id','perfume_id']).copy()
    cols=['user_id','perfume_id','scent_rating','comment_id','is_comment_answer','source_dataset','source_row']
    csv(ratings[cols],out/'user_ratings.csv')
    csv(review[review.scent_rating.eq(0)],audit/'zero_ratings_unresolved.csv')
    csv(review[~review.rating_status.isin(['observed_1_to_10','zero_unresolved'])][['user_id','perfume_id','scent_rating_raw','rating_status','source_row']],audit/'excluded_ratings.csv')
    us=ratings.groupby('user_id').agg(rating_count=('perfume_id','size'),mean_rating=('scent_rating','mean')).reset_index()
    ps=ratings.groupby('perfume_id').agg(rating_count=('user_id','size'),mean_rating=('scent_rating','mean')).reset_index()
    csv(us,audit/'user_rating_counts.csv'); csv(ps,audit/'item_rating_counts.csv')
    acols=['perfume_id','perfume_name','brand','release_year','url','fragrance_group_raw','nature_raw']
    atr=review[acols].drop_duplicates()
    inconsistent=set(atr.loc[atr.duplicated('perfume_id',keep=False),'perfume_id'])
    csv(atr[atr.perfume_id.isin(inconsistent)],audit/'atrafshan_metadata_conflicts.csv')
    atr=atr.drop_duplicates('perfume_id'); csv(atr,out/'atrafshan_perfumes.csv')
    exact,brands=defaultdict(list),defaultdict(list)
    for r in items:
        b,n=r['brand_normalized'],key(r['perfume_name'],r['brand'])
        exact[(b,n)].append(r); brands[b].append((n,r))
    mapping,candidates=[],[]
    for r in atr.to_dict('records'):
        b,n=norm(r['brand']),key(r['perfume_name'],r['brand']); found=exact.get((b,n),[])
        status,accepted,reason='unmatched','','no_exact_brand_and_name'
        if r['perfume_id'] in inconsistent: status,reason='review_required','inconsistent_source_metadata'
        elif len(found)>1: status,reason='review_required','ambiguous_exact_name'
        elif len(found)==1:
            f=found[0]
            if pd.notna(r['release_year']) and f['release_year'] is not None and r['release_year']!=f['release_year']: status,reason='review_required','release_year_conflict'
            else: status,accepted,reason='matched_exact',f['perfume_id'],'unique_normalized_brand_and_name'
        if status!='matched_exact':
            options=[(1.0,f) for f in found]
            if not options:
                options=sorted([(SequenceMatcher(None,n,name).ratio(),f) for name,f in brands.get(b,[])],key=lambda x:(-x[0],x[1]['perfume_id']))[:3]
                options=[x for x in options if x[0]>=0.65]
            for score,f in options:
                candidates.append(dict(atrafshan_id=r['perfume_id'],atrafshan_name=r['perfume_name'],atrafshan_brand=r['brand'],atrafshan_year=r['release_year'],candidate_fragrantica_id=f['perfume_id'],candidate_name=f['perfume_name'],candidate_year=f['release_year'],name_similarity=round(score,4),reason=reason,approved=False))
        mapping.append(dict(atrafshan_id=r['perfume_id'],atrafshan_name=r['perfume_name'],atrafshan_brand=r['brand'],fragrantica_id=accepted,match_status=status,reason=reason))
    mapping=pd.DataFrame(mapping)
    many=mapping.fragrantica_id.ne('') & mapping.fragrantica_id.duplicated(keep=False)
    csv(mapping[many],audit/'mapping_many_to_one_review.csv')
    mapping.loc[many,'match_status']='review_required'; mapping.loc[many,'reason']='multiple_atrafshan_ids_for_one_fragrantica_id'; mapping.loc[many,'fragrantica_id']=''
    csv(mapping,out/'perfume_mapping.csv')
    ccols=['atrafshan_id','atrafshan_name','atrafshan_brand','atrafshan_year','candidate_fragrantica_id','candidate_name','candidate_year','name_similarity','reason','approved']
    csv(pd.DataFrame(candidates,columns=ccols),audit/'mapping_candidates_review.csv')
    matched=mapping[mapping.match_status.eq('matched_exact')]
    linked=ratings[cols].merge(matched[['atrafshan_id','fragrantica_id']],left_on='perfume_id',right_on='atrafshan_id',how='inner',validate='many_to_one').drop(columns='atrafshan_id')
    csv(linked,out/'user_ratings_linked.csv')
    summary=dict(sources=sources,clean_csv_encoding=encoding,fragrantica_clean_rows=len(clean),fragrantica_raw_rows=len(raw),fragrantica_raw_unique_ids=int(raw.perfume_id.nunique()),content_items=len(content),content_eligible_items=int(content.content_eligible.sum()),raw_supplement_matches=int(content.supplement_source_row.ne('').sum()),atrafshan_rows=len(review),atrafshan_users=int(review.loc[review.user_id.ne(''),'user_id'].nunique()),atrafshan_items=len(atr),rating_status_counts=review.rating_status.value_counts().to_dict(),conflicting_rating_pairs=len(conflict_pairs),rating_rows=len(ratings),rating_users=len(us),rating_items=len(ps),users_with_at_least_two_ratings=int(us.rating_count.ge(2).sum()),observed_matrix_density=len(ratings)/(len(us)*len(ps)),mapping_status_counts=mapping.match_status.value_counts().to_dict(),linked_rating_rows=len(linked),linked_rating_users=int(linked.user_id.nunique()),linked_rating_items=int(linked.fragrantica_id.nunique()),mapping_candidates=len(candidates),model_trained=False,split_created=False)
    # Verify exported values and foreign-key relationships, not only in-memory frames.
    co=pd.read_csv(out/'perfumes_content.csv',keep_default_na=False)
    ra=pd.read_csv(out/'user_ratings.csv',keep_default_na=False)
    assert co.perfume_id.is_unique and not ra.duplicated(['user_id','perfume_id']).any()
    assert ra.scent_rating.between(1,10).all()
    assert set(linked.fragrantica_id)<=set(co.perfume_id) and set(ra.perfume_id)<=set(atr.perfume_id)
    assert len(review)==sum(summary['rating_status_counts'].values())
    assert not co.content_text.str.contains(r'\b(?:nan|none|unknown)\b',regex=True).any()
    for col in ['top_notes','middle_notes','base_notes','main_accords']: assert co[col].map(lambda x:isinstance(json.loads(x),list)).all()
    summary['zero_rating_rows_before_id_filter']=int(review.scent_rating.eq(0).sum())
    summary['observed_1_to_10_before_id_filter']=int(review.scent_rating.between(1,10).sum())
    summary['observed_1_to_10_missing_id']=int((review.scent_rating.between(1,10) & review.rating_status.eq('missing_id')).sum())
    summary['validation']='passed_export_roundtrip_ids_rating_range_json_lists_and_count_reconciliation'
    (out/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
    print(json.dumps(summary,ensure_ascii=True,indent=2))
